Brian Saville
August 3, 2026
An attempt to download iNaturalist data to gather a "Bee Dex" csv for Manhattan and the Bronx.

First, import the necessary functions

In [3]:
import requests
from collections import Counter
import pandas as pd
import time

ModuleNotFoundError: No module named 'pandas'

Now, query the iNat database as needed

In [3]:
url = "https://api.inaturalist.org/v2/observations"

bee_id = 630955
bronx_id = 1189
manhattan_id = 1264
num_obs = 13651

params = {
    "taxon_id": bee_id,
    "quality_grade": "research",
    "place_id": 1264,
    "per_page" : 200,
    "fields": "taxon.id,taxon.name,taxon.rank,observed_on"
}


Next, gather these into a dataframe(?)

In [ ]:
all_observations = []
page = 1

#for every page of observations, add results to all_observations list
while page <= 50:
    print(f"Downloading page {page}")

    params["page"] = page

    response = requests.get(url, params=params)
    data = response.json()

    observations = data["results"]

    if len(observations) == 0:
        break

    all_observations.extend(observations)

    page += 1


We'll test if this works by trying to print some of the results.

In [ ]:
for obs in all_observations:
    print(obs["taxon"]["name"])

Success! Let's see if I can use the code made previously to get these into a nice CSV (even though I know it's not yet the full dataset)

In [17]:
#collapsing the species into a checklist
species = set()

for obs in all_observations:
    if obs["taxon"]["rank"] == "species":
        species.add(obs["taxon"]["name"])

print(species)

{'Megachile apicalis', 'Halictus ligatus', 'Andrena vicina', 'Bombus griseocollis', 'Coelioxys octodentatus', 'Calliopsis andreniformis', 'Megachile campanulae', 'Xylocopa virginica', 'Anthidium manicatum', 'Melissodes bimaculatus', 'Xenoglossa pruinosa', 'Bombus perplexus', 'Andrena erigeniae', 'Stelis louisae', 'Agapostemon virescens', 'Megachile sculpturalis', 'Augochlora pura', 'Triepeolus lunatus', 'Coelioxys alternatus', 'Andrena miserabilis', 'Habropoda laboriosa', 'Bombus pensylvanicus', 'Andrena wilkella', 'Bombus impatiens', 'Ptilothrix bombiformis', 'Triepeolus remigatus', 'Melissodes trinodis', 'Megachile rotundata', 'Megachile inimica', 'Lasioglossum coeruleum', 'Bombus citrinus', 'Megachile texana', 'Hylaeus leptocephalus', 'Chelostoma philadelphi', 'Pseudoanthidium nanum', 'Melissodes subillatus', 'Lasioglossum pectorale', 'Lasioglossum imitatum', 'Coelioxys sayi', 'Anthidium oblongatum', 'Osmia georgica', 'Osmia lignaria', 'Andrena milwaukeensis', 'Coelioxys coturnix', 

In [ ]:
#count observations per species
species_counts = Counter()

for obs in all_observations:
    if obs["taxon"]["rank"] == "species":
        species_counts[obs["taxon"]["name"]] += 1

print("\n".join(species_counts))

In [ ]:
#Converting this count into a table
species_df = pd.DataFrame(
    species_counts.items(),
    columns=["Species", "Observations"]
)

print(species_df)

#Sorting that dataframe
species_df = species_df.sort_values(
    by="Observations",
    ascending=False
)

print(species_df)

In [21]:
#Exporting the dataframe
species_df.to_csv("C:/Users/brigu/Documents/_Fordham/_PhD_RESEARCH/bee-project-coding/data/processed/bee_species_test3.csv", index=False)

Okay! That got me a nice CSV summarizing those 10,000 observations. Next steps:

    - get around the 10,000 item limit (or is it a 50-page limit?)
    - get Bronx counts as well
    - merge these into one list, with Manhattan, Bronx, and combined columns (sorted by combined column)
    - maybe even get it to automatically populate it with genus and family columns??

In [5]:
#new query rhat includes observation ID

url = "https://api.inaturalist.org/v2/observations"

bee_id = 630955
bronx_id = 1189
manhattan_id = 1264
num_obs = 13651

params = {
    "taxon_id": bee_id,
    "quality_grade": "research",
    "place_id": 1264,
    "per_page" : 200,
    "order_by": "id",
    "order" : "asc",
    "fields": "id,taxon.id,taxon.name,taxon.rank,observed_on"
}


Now, test if this works by printing the length

In [6]:
response = requests.get(url, params=params)
data = response.json()

observations = data["results"]
print(len(observations))

200


Get the first and last IDs within these observations

In [7]:
print(observations[0]["id"])
print(observations[-1]["id"])

101440
7720049


First: 101440
Second: 7720049

Making another request, using the last ID of the first query as the ID above.

In [9]:
last_id = observations[-1]["id"]

params["id_above"] = last_id

response = requests.get(url, params=params)
data = response.json()

observations_2 = data["results"]
print(len(observations_2))

print(observations_2[0]["id"])
print(observations_2[-1]["id"])

200
7720103
9837936


First: 7720103
Last: 9837936

Now, we can incorporate this trick into a loop to get EVERYTHING

In [ ]:
all_observations = []

params = {
    "taxon_id": bee_id,
    "quality_grade": "research",
    "place_id": 1264,
    "per_page" : 200,
    "order_by": "id",
    "order" : "asc",
    "fields": "id,taxon.id,taxon.name,taxon.rank,observed_on"
}

while True:

    response = requests.get(url, params=params)
    data = response.json()

    observations = data["results"]

    if len(observations) == 0:
        break

    all_observations.extend(observations)

    print("Downloaded: " + str(len(all_observations)) + " observations.")

    params["id_above"] = observations[-1]["id"]

Wow, okay. Now let's see about outputting them into a checklist. This code can be copy/pasted from above:

In [ ]:
#collapsing the species into a checklist
species = set()

for obs in all_observations:
    if obs["taxon"]["rank"] == "species":
        species.add(obs["taxon"]["name"])

print(species)

#count observations per species
species_counts = Counter()

for obs in all_observations:
    if obs["taxon"]["rank"] == "species":
        species_counts[obs["taxon"]["name"]] += 1

print("\n".join(species_counts))

#Converting this count into a table
species_df = pd.DataFrame(
    species_counts.items(),
    columns=["Species", "Observations"]
)

print(species_df)

#Sorting that dataframe
species_df = species_df.sort_values(
    by="Observations",
    ascending=False
)

print(species_df)

#Exporting the dataframe
species_df.to_csv("C:/Users/brigu/Documents/_Fordham/_PhD_RESEARCH/bee-project-coding/data/processed/bee_species_checklist.csv", index=False)

Next, I want to grab genus, family, and common name to add to my CSV. First, I need to see how these bits of information are being stored.

In [1]:
url = "https://api.inaturalist.org/v1/taxa/39682"

response = requests.get(url)
taxon_data = response.json()

print(taxon_data)

NameError: name 'requests' is not defined